# Bac a sable — pandas + SQL

- **ZONE SETUP** : les donnees, a lancer une fois. Reutilisables partout (pandas: `commandes/clients/produits` ; SQL: `q("...")`).
- **ZONE VIDE** : tu ecris ici le code des exercices en cours (enonces en commentaire). On corrige jusqu'a ce que ce soit bon.
- **ARCHIVES** (en bas) : exos reussis, regroupes par THEME + DATE.

Aide-memoire : `fiches-revision/Carnet_de_recettes.md`.

## ZONE SETUP — executer une fois

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

commandes = pd.DataFrame({
    "client":   ["Alice","Alice","Bob","Bob","Chloe","Chloe","David","Alice","Bob","David"],
    "produit":  ["P1","P2","P1","P3","P2","P3","P1","P3","P2","P2"],
    "quantite": [2, 1, 5, 3, 2, 1, 4, np.nan, 2, 1],
    "prix_unitaire": [10.0, 30.0, 10.0, 100.0, 30.0, 100.0, 10.0, 100.0, 30.0, 30.0],
    "date":     ["2025-01-05","2025-01-07","2025-01-09","2025-02-01","2025-02-03",
                 "2025-02-10","2025-03-01","2025-03-05","2025-03-08","2025-03-20"],
})
commandes = pd.concat([commandes, commandes.iloc[[2]]], ignore_index=True)  # 1 doublon expres

clients = pd.DataFrame({
    "client":  ["Alice","Bob","Chloe","David"],
    "ville":   ["Paris","Lyon","Paris","Marseille"],
    "segment": ["Pro","Particulier","Pro","Particulier"],
})

produits = pd.DataFrame({
    "produit":   ["P1","P2","P3"],
    "categorie": ["Accessoire","Accessoire","Tech"],
    "prix_catalogue": [10.0, 30.0, 100.0],
})

conn = sqlite3.connect(":memory:")
commandes.to_sql("commandes", conn, index=False, if_exists="replace")
clients.to_sql("clients", conn, index=False, if_exists="replace")
produits.to_sql("produits", conn, index=False, if_exists="replace")

def q(sql):
    "Execute une requete SQL et renvoie un DataFrame."
    return pd.read_sql_query(sql, conn)

print("OK - commandes / clients / produits prets  |  SQL: q(\"SELECT ...\")")
commandes

## ZONE VIDE — tes exercices du jour (ecris sous chaque enonce)

In [ ]:
# [SQL] Exo 1 — HAVING
# Les villes dont le CA total depasse 200 EUR.
# Indice : JOIN clients + GROUP BY ville + HAVING SUM(...) > 200.
q("""
  SELECT cl.ville, SUM(co.prix_unitaire*co.quantite)
  FROM commandes co
  join clients cl on co.client=cl.client
  
  """)


In [ ]:
# [SQL] Exo 2 — ROW_NUMBER : la commande la plus chere de CHAQUE client
# montant d'une ligne = quantite * prix_unitaire.
# Indice : une CTE avec ROW_NUMBER() OVER (PARTITION BY client ORDER BY montant DESC), puis WHERE rang = 1.



In [ ]:
# [pandas] Exo 3 — top produit (par CA) dans CHAQUE categorie
# Indice : merge produits -> CA -> groupby(["categorie","produit"]).sum -> trier -> groupby("categorie").head(1).



In [ ]:
# [pandas] Exo 4 — API -> pandas (json_normalize)
data = [
    {"id": 1, "client": {"nom": "Alice", "ville": "Paris"}, "total": 120},
    {"id": 2, "client": {"nom": "Bob",   "ville": "Lyon"},  "total": 80},
]
# Transforme "data" en DataFrame APLATI avec pd.json_normalize(...)



---
# ARCHIVES — exos reussis, regroupes par THEME + DATE

**Convention :**
- 1 seul theme dans la journee -> un titre `THEME — DATE`, puis les cellules d'exos dessous.
- Plusieurs themes -> un titre `THEMES — DATE`, puis des sous-titres `• Theme X — DATE` avec leurs cellules.
- Sous chaque theme : `A re-tester le ...` (date + 7 jours) = ton calendrier de repetition espacee.

## pandas — regrouper & rapprocher (groupby / merge / agg) — 03/09/2026
**A re-tester le 10/09/2026**.

In [ ]:
# Exo : CA total par client, du + gros au + petit, top 3.
commandes = commandes.dropna()                                    # (remarque: modifie la table de base + inutile ici, .sum() ignore les NaN)
print(commandes)
commandes["CA"] = commandes["quantite"] * commandes["prix_unitaire"]
commandes.groupby("client")["CA"].sum().nlargest(3)

In [ ]:
# Exo : CA total par categorie (merge commandes + produits).
commande_produit = commandes.merge(produits, on="produit")
commande_produit["CA"] = commande_produit["prix_unitaire"] * commande_produit["quantite"]
commande_produit.groupby("categorie")["CA"].sum()

In [ ]:
# Exo : par ville, nb de commandes + CA total.
commande_client = commandes.merge(clients, on="client")
commande_client["CA"] = commande_client["prix_unitaire"] * commande_client["quantite"]
commande_client = commande_client.groupby("ville").agg(
    nb_commande=("quantite", "count"),   # count ignore les NaN ; ("client","count") ou .size comptent TOUT
    CA_tot=("CA", "sum")
)
commande_client
# Amelioration : trier -> .sort_values("CA_tot", ascending=False)

## THEMES — 04/09/2026
**A re-tester le 11/09/2026**.

### • pandas — pivot_table / apply / panier moyen — 04/09/2026

In [ ]:
# Exo : pivot_table CA par ville (lignes) x categorie (colonnes).
commandes_client = commandes.merge(clients, on="client") \
                            .merge(produits, on="produit") \
                            .assign(CA=lambda x: x["prix_unitaire"]*x["quantite"])
commandes_client.pivot_table(index="ville", columns="categorie", values="CA", aggfunc="sum") \
                .fillna("Categorie indisponible dans cette ville")
# Amelioration propre : fill_value=0 DANS pivot_table (garde du numerique) au lieu du fillna texte.

In [ ]:
# Exo : colonne "gamme" = premium si prix_unitaire >= 100 sinon standard.
commandes["gamme"] = commandes.apply(lambda x: "premium" if x["prix_unitaire"] >= 100 else "standard", axis=1)
commandes
# Variante plus rapide (1 seule colonne) : np.where(commandes["prix_unitaire"]>=100, "premium", "standard")

In [ ]:
# Exo : panier moyen (CA moyen par commande) par segment.
commandes_client_produit = commandes.assign(CA=lambda d: d.prix_unitaire * d.quantite) \
                                    .merge(clients, on="client") \
                                    .merge(produits, on="produit")
commandes_client_produit = commandes_client_produit.groupby("segment")["CA"].mean()
display(commandes_client_produit)
# Note : le merge(produits) est inutile ici (categorie pas utilisee), mais sans consequence.

### • SQL — window & CTE (RANK, cumul, CTE) — 04/09/2026

In [ ]:
# Exo : classer les clients par CA total (RANK). Ta 1re fonction fenetre ecrite !
q("""
  SELECT client, SUM(quantite*prix_unitaire), RANK() over (order by SUM(quantite*prix_unitaire) desc)
  FROM commandes co
  GROUP BY client
  """)
# Amelioration entretien : nommer les colonnes -> ... AS ca, ... AS rang.

In [ ]:
# Exo : cumul du CA par date (running total).
q("""
  SELECT date,
         SUM(prix_unitaire*quantite) as CA,
         SUM(SUM(prix_unitaire * quantite)) over (ORDER BY date) as CA_cumulee
  FROM commandes
  GROUP BY date
  """)
# Cle : DEUX etages. SUM interne = total du jour (GROUP BY) ; SUM(...) OVER externe = cumul de ces totaux.

In [ ]:
# Exo : clients dont le CA total depasse le CA moyen des clients (CTE).
q("""
  WITH calcul_CA_moyen AS (
      SELECT co.client, SUM(co.prix_unitaire * co.quantite) as CA_tot
      FROM commandes co
      group by co.client              -- la ligne qui manquait : sans elle, SUM agrege TOUT en 1 ligne
  )
  SELECT *
  FROM calcul_CA_moyen
  WHERE CA_tot > (SELECT AVG(CA_tot) from calcul_CA_moyen)
  """)